# 1. Load Bronze tables

In [0]:
table_names = ['orders', 'customers', 'order_items', 'products']

In [0]:
bronze_tables = {}

for table_name in table_names:
    bronze_tables[table_name] = spark.table('olist.bronze.' + table_name)

In [0]:
bronze_tables['orders'].display()

# 2. Data quality checks

## 1. check row count

In [0]:
for name, df in bronze_tables.items():
    print(f"Bronze {name} table row count: {df.count()}")

## 2. check schema

In [0]:
for name, df in bronze_tables.items():
    print(name)
    df.printSchema()
    print()

## 3. Check null values

### 3.1 Create a helper function

In [0]:
from pyspark.sql import functions as F

In [0]:
# how to filter for a specific one null column
(
    bronze_tables['orders']
    .filter(F.col("order_delivered_customer_date").isNull())
    .display()
)

In [0]:
# count how many null values there are in one column
bronze_tables['orders'].agg(
    # if the column is null, then set 1 to the value
    # then add all 1s (null values)
    F.sum(F.when(F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("null count order_delivered_customer_date")
).display()

In [0]:
bronze_tables['orders'].columns

In [0]:
# count how many null values there are in more columns

bronze_tables['orders'].agg(
    # if the column is null, then set 1 to the value
    # then add all 1s (null values)
    F.sum(F.when(F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("order_delivered_customer_date"),
    F.sum(F.when(F.col("order_delivered_carrier_date").isNull(), 1).otherwise(0)).alias("order_delivered_carrier_date"),
    F.sum(F.when(F.col("order_estimated_delivery_date").isNull(), 1).otherwise(0)).alias("order_estimated_delivery_date")
).display()

In [0]:
# count how many null values there are in all columns
null_expressions = []

for c in bronze_tables['orders'].columns:
    # using the same expression as before, but this time the column name is changed dynamically
    expression = F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)

    null_expressions.append(expression)

bronze_tables['orders'].agg(*null_expressions).display()


In [0]:
# make a function of the code above. it will return a dataframe with null counts by columns
def get_null_counts(df):
    # count how many null values there are in all columns
    null_expressions = []

    for c in df.columns:
        # using the same expression as before, but this time the column name is changed dynamically
        expression = F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)

        null_expressions.append(expression)

    return df.agg(*null_expressions)

### 3.2 check Orders table

In [0]:
# check if the function returns the same result
get_null_counts(bronze_tables['orders']).display()

In [0]:
# there's a lot of null values in 'order delivered customer date' check why
(
    bronze_tables['orders']
    .filter('order_delivered_customer_date is null')
    .groupby('order_status')
    .count()
    .sort("count", ascending=False)
    .display()
)
# many orders were shipped and not delivered
# canceled and unavailable make sense
# invoiced and processing means that the order was not even shipped yet
# 8 delivered orders have no delivery timestamp (potential data quality issue)

In [0]:
(
    bronze_tables['orders']
    .filter('order_delivered_customer_date is null and order_status="delivered"')
    .display()
 )

### 3.3 check Customers table

In [0]:
# there are no null values in customers table
get_null_counts(bronze_tables['customers']).display()

### 3.4 check order_items table

In [0]:
# no null values in order_items table
get_null_counts(bronze_tables['order_items']).display()

### 3.5 check products table

In [0]:
# there are null values in the product table:
# 610 nulls in product_category_name, product_name_lenght, product_description_lenght, product_photos_qty
# 2 nulls in product_weight_g, product_lenght_cm, product_height_cm, product_witdh_cm
get_null_counts(bronze_tables['products']).display()

In [0]:
# check rows with null product_name_lenght
# all text fields are null, but the numerical data is present
(
    bronze_tables['products']
    .filter(F.col('product_name_lenght').isNull())
    .display()
)

In [0]:
# check rows with null product_length_cm
# all text fields are null, but the numerical data is present
(
    bronze_tables['products']
    .filter(F.col('product_length_cm').isNull())
    .display()
)

## 4. Check for duplicates

### 4.1 create a helper function

In [0]:
## check how many times each order_status appears in orders - this should be an easy example to verify if the function returns a value higher than 1
(
    bronze_tables['orders']
    .groupby('order_status')
    .count()
    .sort("count", ascending=False)
    .display()
)


In [0]:
# we can filter for cases when count is higher than 10
(
    bronze_tables['orders']
    .groupby('order_status')
    .count()
    .filter('count > 10')
    .sort("count", ascending=False)
    .display()
)

In [0]:
# create a helper function, returning only the duplicated values for dataframe and column combination
# the function returns a dataframe

def get_duplicated_values(df, column):
    return (
        df
        .groupby(column)
        .count()
        .filter('count > 1')
        .sort("count", ascending=False)
        )

In [0]:
# check if the result is similar to above

get_duplicated_values(bronze_tables['orders'], 'order_status').display()

### 4.2 check order table for duplicated ids

In [0]:
# list all column names
bronze_tables['orders'].columns

In [0]:
# no duplicates in order_id
get_duplicated_values(bronze_tables['orders'], 'order_id').display()

In [0]:
# no duplicates in customer_id
get_duplicated_values(bronze_tables['orders'], 'customer_id').display()

### 4.3 check customers table for duplicated ids

In [0]:
bronze_tables['customers'].columns

In [0]:
# no duplicates
get_duplicated_values(bronze_tables['customers'], 'customer_id').display()

In [0]:
# customer_unique_id is duplicated
get_duplicated_values(bronze_tables['customers'], 'customer_unique_id').display()

In [0]:
# check the customer_unique_id that appeared 17 times
(
    bronze_tables['customers']
    .filter('customer_unique_id == "8d50f5eadf50201ccdcedfb9e2ac8455"')
    .display()
 )

In [0]:
(
    bronze_tables['customers']
    .filter('customer_unique_id == "8d50f5eadf50201ccdcedfb9e2ac8455"')
    .groupby(['customer_zip_code_prefix', 'customer_city', 'customer_state'])
    .count()
    .display()
)

In [0]:
# orders table uses customer_id, not the customer unique id (from the customers table)
#filter using customer unique id:
(
    bronze_tables['orders']
    .filter('customer_id == "8d50f5eadf50201ccdcedfb9e2ac8455"')
    .display()

)

In [0]:
# filter using customer_id from the customers table
(
    bronze_tables['orders']
    .filter('customer_id == "1bd3585471932167ab72a84955ebefea"')
    .display()

)


`customer_id` is an order-level customer identifier used to join with the `orders` table.

`customer_unique_id` identifies the same customer across multiple orders.

Duplicate `customer_unique_id` values are expected and should not be removed.

The advantage of this model is that `customer_unique_id` identifies the same user across multiple orders, while each `customer_id` can store location information associated with a specific order.

If we removed duplicate `customer_unique_id` values, we could lose location information at the order level.

In [0]:
bronze_tables['customers'].columns

In [0]:
# check if there's any customer_unique_id with more than one location
(
bronze_tables['customers']
    .groupby('customer_unique_id')
    .agg(F.count_distinct('customer_zip_code_prefix',
        'customer_city',
        'customer_state'
        ).alias('count_distinct'))
    .filter("count_distinct > 1")
    .sort('count_distinct', ascending=False)
    .display()
)


In [0]:
# check the customer with 3 different locations
(
    bronze_tables['customers']
    .filter('customer_unique_id == "3e43e6105506432c953e165fb2acf44c"')
    .display()
)

### 4.4 check products table for duplicated ids

In [0]:
bronze_tables['products'].display()

In [0]:
# there are no duplicated product_ids
get_duplicated_values(bronze_tables['products'], 'product_id').display()

### 4.5 check order_items table for duplicated ids

In [0]:
bronze_tables['order_items'].display()

In [0]:
# order_id is duplicated
get_duplicated_values(bronze_tables['order_items'], 'order_id').display()

In [0]:
# order_item_id is duplicated
get_duplicated_values(bronze_tables['order_items'], 'order_item_id').display()

In [0]:
# check if the combination order_id + order_item_id is unique
(
    bronze_tables['order_items']
    .groupBy(['order_id', 'order_item_id'])
    .count()
    .filter('count > 1')
    .display()
)

## Data quality summary

- Row counts and schemas were reviewed for all Bronze tables.
- Null values were checked across all datasets.
- Missing delivery timestamps in `orders` are mostly consistent with order status.
- 8 delivered orders have no delivery timestamp and are kept unchanged for now.
- `products` contains missing product metadata that will be handled intentionally in the cleaning step.
- No duplicate business keys were found:
  - `orders`: `order_id`
  - `customers`: `customer_id`
  - `products`: `product_id`
  - `order_items`: (`order_id`, `order_item_id`)
- Duplicate `customer_unique_id` values are expected because one customer can place multiple orders.
- The same customer may have different location records across orders.